# MLOps Training 2026/2027 — Task 2
## Notebook 6: Train, Tune, and Evaluate
In this notebook, I will train a simple baseline model first, then train and tune a classification model using the training and validation sets.

Because late deliveries are the minority class, I will evaluate models using precision, recall, F1-score, and the confusion matrix instead of relying on accuracy alone.

The test set will only be used once at the very end for the final evaluation.

## 1. Load the Model Datasets
I will load the final feature tables and target values created in Notebook 5.

In [1]:
import pandas as pd
import numpy as np

X_train = pd.read_csv("artifacts/X_train.csv")
X_validation = pd.read_csv("artifacts/X_validation.csv")
X_test = pd.read_csv("artifacts/X_test.csv")

y_train = pd.read_csv("artifacts/y_train.csv").squeeze("columns")
y_validation = pd.read_csv("artifacts/y_validation.csv").squeeze("columns")
y_test = pd.read_csv("artifacts/y_test.csv").squeeze("columns")

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_validation.shape, y_validation.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (67533, 64) (67533,)
Validation: (14471, 64) (14471,)
Test: (14472, 64) (14472,)


## 2. Baseline Model
I will start with a simple baseline that predicts every order as the majority class.

This gives a minimum performance level that the trained model should improve on.

In [2]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

baseline_predictions = np.zeros(len(y_validation), dtype=int)

baseline_accuracy = accuracy_score(
    y_validation,
    baseline_predictions
)

baseline_precision = precision_score(
    y_validation,
    baseline_predictions,
    zero_division=0
)

baseline_recall = recall_score(
    y_validation,
    baseline_predictions,
    zero_division=0
)

baseline_f1 = f1_score(
    y_validation,
    baseline_predictions,
    zero_division=0
)

print("Baseline Accuracy:", round(baseline_accuracy, 4))
print("Baseline Precision:", round(baseline_precision, 4))
print("Baseline Recall:", round(baseline_recall, 4))
print("Baseline F1:", round(baseline_f1, 4))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        baseline_predictions
    )
)

Baseline Accuracy: 0.9569
Baseline Precision: 0.0
Baseline Recall: 0.0
Baseline F1: 0.0

Confusion Matrix:
[[13847     0]
 [  624     0]]


### 2.1 Baseline Interpretation
The baseline achieves high accuracy because most orders are on time.

However, it fails to identify any late deliveries. Its precision, recall, and F1-score for the late class are all 0.

This confirms that accuracy alone is not an appropriate metric for this problem.

## 3. Logistic Regression
I will train a logistic regression model using class balancing so that the minority late-delivery class receives more weight during training.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train,
    y_train
)

validation_predictions = logistic_model.predict(
    X_validation
)

print(
    classification_report(
        y_validation,
        validation_predictions,
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        validation_predictions
    )
)

              precision    recall  f1-score   support

           0     0.9806    0.6986    0.8159     13847
           1     0.0940    0.6939    0.1656       624

    accuracy                         0.6984     14471
   macro avg     0.5373    0.6962    0.4907     14471
weighted avg     0.9424    0.6984    0.7879     14471

Confusion Matrix:
[[9673 4174]
 [ 191  433]]


### 3.1 Logistic Regression Interpretation
The balanced logistic regression identifies many more late deliveries than the baseline.

It correctly detects 433 of the 624 late orders, giving a recall of about 69.39%. However, precision is low at about 9.40%, which means many orders predicted as late are actually on time.

This shows a trade-off between detecting more late orders and creating more false positives. The model will be tuned using the validation set to improve this balance.

## 4. Tune the Classification Threshold
The default classification threshold is 0.50. I will test different thresholds on the validation set and choose the one that gives the best F1-score for late deliveries.

In [4]:
validation_probabilities = logistic_model.predict_proba(
    X_validation
)[:, 1]

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):
    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results

,threshold,precision,recall,f1
0,0.10,0.044055,1.000000,0.084393
1,0.15,0.045306,1.000000,0.086685
2,0.20,0.047171,0.996795,0.090080
3,0.25,0.050356,0.987179,0.095823
4,0.30,0.054728,0.971154,0.103616
5,0.35,0.060768,0.945513,0.114197
6,0.40,0.069142,0.889423,0.128309
7,0.45,0.079751,0.799679,0.145037
8,0.50,0.093987,0.693910,0.165552
9,0.55,0.110205,0.584936,0.185467


### 4.1 Threshold Selection
The best validation F1-score was achieved at a threshold of 0.70.

At this threshold, precision increased compared with the default 0.50 threshold, while recall decreased. The overall F1-score improved, so I will use 0.70 as the selected threshold for the logistic regression model.

In [5]:
best_threshold = 0.70

tuned_validation_predictions = (
    validation_probabilities >= best_threshold
).astype(int)

print(
    classification_report(
        y_validation,
        tuned_validation_predictions,
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        tuned_validation_predictions
    )
)

              precision    recall  f1-score   support

           0     0.9683    0.9280    0.9477     13847
           1     0.1699    0.3269    0.2236       624

    accuracy                         0.9021     14471
   macro avg     0.5691    0.6275    0.5857     14471
weighted avg     0.9339    0.9021    0.9165     14471

Confusion Matrix:
[[12850   997]
 [  420   204]]


### 4.2 Tuned Logistic Regression Result
Using a threshold of 0.70 improved the late-delivery F1-score to 0.2236.

The model now produces fewer false positive late alerts than at the default threshold, although recall is lower. This threshold gives a better balance between precision and recall according to the validation F1-score.

## 5. Random Forest Model
I will train a Random Forest classifier as a second model and compare its validation performance with logistic regression.

In [6]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest.fit(
    X_train,
    y_train
)

rf_validation_predictions = random_forest.predict(
    X_validation
)

print(
    classification_report(
        y_validation,
        rf_validation_predictions,
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        rf_validation_predictions
    )
)

              precision    recall  f1-score   support

           0     0.9667    0.9208    0.9432     13847
           1     0.1444    0.2965    0.1942       624

    accuracy                         0.8939     14471
   macro avg     0.5556    0.6087    0.5687     14471
weighted avg     0.9313    0.8939    0.9109     14471

Confusion Matrix:
[[12751  1096]
 [  439   185]]


### 5.1 Random Forest Result
The Random Forest achieved a late-delivery F1-score of 0.1942 on the validation set.

Its performance was better than the majority-class baseline, but its late-delivery F1-score was lower than the tuned logistic regression score of 0.2236.

Therefore, logistic regression remains the better model based on the selected validation metric.

## 6. Model Comparison
I will compare the baseline, logistic regression, and Random Forest using validation performance. The final model will be selected before evaluating on the test set.

In [7]:
model_comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression - Default Threshold",
        f"Logistic Regression - Threshold {best_threshold:.2f}",
        "Random Forest"
    ],
    "Precision": [
        baseline_precision,
        precision_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        precision_score(
            y_validation,
            tuned_validation_predictions,
            zero_division=0
        ),
        precision_score(
            y_validation,
            rf_validation_predictions,
            zero_division=0
        )
    ],
    "Recall": [
        baseline_recall,
        recall_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        recall_score(
            y_validation,
            tuned_validation_predictions,
            zero_division=0
        ),
        recall_score(
            y_validation,
            rf_validation_predictions,
            zero_division=0
        )
    ],
    "F1": [
        baseline_f1,
        f1_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        f1_score(
            y_validation,
            tuned_validation_predictions,
            zero_division=0
        ),
        f1_score(
            y_validation,
            rf_validation_predictions,
            zero_division=0
        )
    ]
})

model_comparison.round(4)

,Model,Precision,Recall,F1
0,Baseline,0.0000,0.0000,0.0000
1,Logistic Regression - Default Threshold,0.0940,0.6939,0.1656
2,Logistic Regression - Threshold 0.70,0.1699,0.3269,0.2236
3,Random Forest,0.1444,0.2965,0.1942


### 6.1 Final Model Selection
Based on validation F1-score, the logistic regression model with a classification threshold of 0.70 performed best.

This model and threshold will now be used for the final test evaluation.

## 7. Final Test Evaluation
The final model will now be evaluated on the test set. The test set has not been used for model selection or tuning.

In [8]:
test_probabilities = logistic_model.predict_proba(
    X_test
)[:, 1]

final_test_predictions = (
    test_probabilities >= best_threshold
).astype(int)

print(
    classification_report(
        y_test,
        final_test_predictions,
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        final_test_predictions
    )
)

              precision    recall  f1-score   support

           0     0.9619    0.8510    0.9031     13852
           1     0.0690    0.2468    0.1079       620

    accuracy                         0.8251     14472
   macro avg     0.5155    0.5489    0.5055     14472
weighted avg     0.9236    0.8251    0.8690     14472

Confusion Matrix:
[[11788  2064]
 [  467   153]]


### 7.1 Final Test Result
On the unseen test set, the final logistic regression model achieved a late-delivery precision of 0.0690, recall of 0.2468, and F1-score of 0.1079.

The test performance was lower than the validation performance. Since the data was split chronologically, this suggests that delivery patterns changed over time and the model did not generalize as strongly to the later test period.

The model still performs better than the majority-class baseline for identifying late deliveries, because the baseline detects no late orders at all.

## 8. Save Final Model and Results
I will save the selected logistic regression model, its classification threshold, and a short summary of the final results.

In [9]:
import joblib
import json

joblib.dump(
    logistic_model,
    "artifacts/final_logistic_regression.joblib"
)

final_results = {
    "model": "Logistic Regression",
    "threshold": float(best_threshold),
    "validation_f1": float(
        f1_score(
            y_validation,
            tuned_validation_predictions,
            zero_division=0
        )
    ),
    "test_precision_late": float(
        precision_score(
            y_test,
            final_test_predictions,
            zero_division=0
        )
    ),
    "test_recall_late": float(
        recall_score(
            y_test,
            final_test_predictions,
            zero_division=0
        )
    ),
    "test_f1_late": float(
        f1_score(
            y_test,
            final_test_predictions,
            zero_division=0
        )
    ),
    "test_accuracy": float(
        accuracy_score(
            y_test,
            final_test_predictions
        )
    )
}

with open(
    "artifacts/final_model_results.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_results,
        file,
        indent=4
    )

print("Final model and results saved successfully!")

Final model and results saved successfully!


## 9. Conclusion
A majority-class baseline, logistic regression, and Random Forest model were compared using the validation set.

The tuned logistic regression model achieved the best validation F1-score for late deliveries, so it was selected as the final model.

The final model was then evaluated once on the unseen test set. Its test performance was lower than its validation performance, which suggests that the model may be sensitive to changes in delivery patterns over time.

The trained model, selected threshold, preprocessing artifacts, feature list, and result summary were saved for future use.